In [1]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Same config as your main model for fair comparison
LABEL_PERCENTAGE = 0.20  # 20% labeled data

# --- 1. Data Loading & Feature Selection ---
def load_stats_only():
    print("--- Loading Merged CSV (Stats Only) ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # --- KEY STEP: SELECT ONLY STATS COLUMNS ---
    # We explicitly EXCLUDE 'alpha_' (Payload) and label/metadata columns
    exclude_keywords = ['alpha_', 'application', 'category', 'binary_type', 'filename']

    # Logic: Keep columns that start with beta/gamma/fft AND are not in exclude list
    stats_cols = [c for c in df.columns if
                  (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
                  and not any(x in c for x in exclude_keywords)]

    print(f"Selected {len(stats_cols)} Statistical Features (Beta/Gamma/FFT).")
    print(f"Skipped Alpha (Payload) features.")

    X_stats = df[stats_cols].values.astype('float32')

    # Normalize (Standard practice for XGBoost to converge faster, though not strictly required)
    print("Normalizing Statistical Features...")
    X_stats = StandardScaler().fit_transform(X_stats)

    return df, X_stats

# --- 2. Generic Trainer (Same as before) ---
def train_task(X, y, task_name):
    print(f"\n>>> Starting Baseline Task: {task_name}")
    print(f"    Data Shape: {X.shape}")

    # 1. Split (Same 20% split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # 2. Train (Same XGBoost Hyperparameters)
    # We use the exact same settings to ensure the difference comes from DATA, not the model.
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    # 3. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

# --- 3. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X_stats = load_stats_only()
    if df_raw is None: return

    # 2. Reconstruct Labels (VPN_Prefixing)
    # We need to rebuild the labels exactly as we did for the main model
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # The Input for all tasks is just the Stats
    X_final = X_stats

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 1: BINARY DETECTION (Stats Only)")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task(X_final, y_bin, "Binary Baseline")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))


    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 2: VPN CATEGORY (Stats Only)")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task(X_cat, y_cat, "VPN Category Baseline")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))


    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 3: VPN TOP 6 APPS (Stats Only)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task(X_app, y_app, "VPN Top Apps Baseline")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV (Stats Only) ---
Selected 137 Statistical Features (Beta/Gamma/FFT).
Skipped Alpha (Payload) features.
Normalizing Statistical Features...
Reconstructing Labels...

 BASELINE 1: BINARY DETECTION (Stats Only)

>>> Starting Baseline Task: Binary Baseline
    Data Shape: (12555, 137)
    Train: 2511 | Test: 10044
    >>> Binary Baseline Weighted F1: 0.9543
              precision    recall  f1-score   support

     Non-VPN       0.98      0.96      0.97      7843
         VPN       0.86      0.94      0.90      2201

    accuracy                           0.95     10044
   macro avg       0.92      0.95      0.93     10044
weighted avg       0.96      0.95      0.95     10044


 BASELINE 2: VPN CATEGORY (Stats Only)

>>> Starting Baseline Task: VPN Category Baseline
    Data Shape: (2751, 137)
    Train: 550 | Test: 2201
    >>> VPN Category Baseline Weighted F1: 0.8834
                   precision    recall  f1-score   support

         VPN_Chat       0.72      0.6

In [5]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Same config as your main model for fair comparison
LABEL_PERCENTAGE = 0.05  # 20% labeled data

# --- 1. Data Loading & Feature Selection ---
def load_stats_only():
    print("--- Loading Merged CSV (Stats Only) ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # --- KEY STEP: SELECT ONLY STATS COLUMNS ---
    # We explicitly EXCLUDE 'alpha_' (Payload) and label/metadata columns
    exclude_keywords = ['alpha_', 'application', 'category', 'binary_type', 'filename']

    # Logic: Keep columns that start with beta/gamma/fft AND are not in exclude list
    stats_cols = [c for c in df.columns if
                  (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
                  and not any(x in c for x in exclude_keywords)]

    print(f"Selected {len(stats_cols)} Statistical Features (Beta/Gamma/FFT).")
    print(f"Skipped Alpha (Payload) features.")

    X_stats = df[stats_cols].values.astype('float32')

    # Normalize (Standard practice for XGBoost to converge faster, though not strictly required)
    print("Normalizing Statistical Features...")
    X_stats = StandardScaler().fit_transform(X_stats)

    return df, X_stats

# --- 2. Generic Trainer (Same as before) ---
def train_task(X, y, task_name):
    print(f"\n>>> Starting Baseline Task: {task_name}")
    print(f"    Data Shape: {X.shape}")

    # 1. Split (Same 20% split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # 2. Train (Same XGBoost Hyperparameters)
    # We use the exact same settings to ensure the difference comes from DATA, not the model.
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    # 3. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

# --- 3. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X_stats = load_stats_only()
    if df_raw is None: return

    # 2. Reconstruct Labels (VPN_Prefixing)
    # We need to rebuild the labels exactly as we did for the main model
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # The Input for all tasks is just the Stats
    X_final = X_stats

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 1: BINARY DETECTION (Stats Only)")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task(X_final, y_bin, "Binary Baseline")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))


    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 2: VPN CATEGORY (Stats Only)")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task(X_cat, y_cat, "VPN Category Baseline")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))


    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 3: VPN TOP 6 APPS (Stats Only)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task(X_app, y_app, "VPN Top Apps Baseline")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV (Stats Only) ---
Selected 137 Statistical Features (Beta/Gamma/FFT).
Skipped Alpha (Payload) features.
Normalizing Statistical Features...
Reconstructing Labels...

 BASELINE 1: BINARY DETECTION (Stats Only)

>>> Starting Baseline Task: Binary Baseline
    Data Shape: (12555, 137)
    Train: 627 | Test: 11928
    >>> Binary Baseline Weighted F1: 0.9376
              precision    recall  f1-score   support

     Non-VPN       0.97      0.95      0.96      9314
         VPN       0.84      0.88      0.86      2614

    accuracy                           0.94     11928
   macro avg       0.90      0.92      0.91     11928
weighted avg       0.94      0.94      0.94     11928


 BASELINE 2: VPN CATEGORY (Stats Only)

>>> Starting Baseline Task: VPN Category Baseline
    Data Shape: (2751, 137)
    Train: 137 | Test: 2614
    >>> VPN Category Baseline Weighted F1: 0.7595
                   precision    recall  f1-score   support

         VPN_Chat       0.39      0.25

In [6]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Same config as your main model for fair comparison
LABEL_PERCENTAGE = 0.10  # 20% labeled data

# --- 1. Data Loading & Feature Selection ---
def load_stats_only():
    print("--- Loading Merged CSV (Stats Only) ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # --- KEY STEP: SELECT ONLY STATS COLUMNS ---
    # We explicitly EXCLUDE 'alpha_' (Payload) and label/metadata columns
    exclude_keywords = ['alpha_', 'application', 'category', 'binary_type', 'filename']

    # Logic: Keep columns that start with beta/gamma/fft AND are not in exclude list
    stats_cols = [c for c in df.columns if
                  (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
                  and not any(x in c for x in exclude_keywords)]

    print(f"Selected {len(stats_cols)} Statistical Features (Beta/Gamma/FFT).")
    print(f"Skipped Alpha (Payload) features.")

    X_stats = df[stats_cols].values.astype('float32')

    # Normalize (Standard practice for XGBoost to converge faster, though not strictly required)
    print("Normalizing Statistical Features...")
    X_stats = StandardScaler().fit_transform(X_stats)

    return df, X_stats

# --- 2. Generic Trainer (Same as before) ---
def train_task(X, y, task_name):
    print(f"\n>>> Starting Baseline Task: {task_name}")
    print(f"    Data Shape: {X.shape}")

    # 1. Split (Same 20% split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # 2. Train (Same XGBoost Hyperparameters)
    # We use the exact same settings to ensure the difference comes from DATA, not the model.
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    # 3. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

# --- 3. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X_stats = load_stats_only()
    if df_raw is None: return

    # 2. Reconstruct Labels (VPN_Prefixing)
    # We need to rebuild the labels exactly as we did for the main model
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # The Input for all tasks is just the Stats
    X_final = X_stats

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 1: BINARY DETECTION (Stats Only)")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task(X_final, y_bin, "Binary Baseline")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))


    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 2: VPN CATEGORY (Stats Only)")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task(X_cat, y_cat, "VPN Category Baseline")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))


    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 3: VPN TOP 6 APPS (Stats Only)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task(X_app, y_app, "VPN Top Apps Baseline")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV (Stats Only) ---
Selected 137 Statistical Features (Beta/Gamma/FFT).
Skipped Alpha (Payload) features.
Normalizing Statistical Features...
Reconstructing Labels...

 BASELINE 1: BINARY DETECTION (Stats Only)

>>> Starting Baseline Task: Binary Baseline
    Data Shape: (12555, 137)
    Train: 1255 | Test: 11300
    >>> Binary Baseline Weighted F1: 0.9445
              precision    recall  f1-score   support

     Non-VPN       0.97      0.95      0.96      8824
         VPN       0.85      0.91      0.88      2476

    accuracy                           0.94     11300
   macro avg       0.91      0.93      0.92     11300
weighted avg       0.95      0.94      0.94     11300


 BASELINE 2: VPN CATEGORY (Stats Only)

>>> Starting Baseline Task: VPN Category Baseline
    Data Shape: (2751, 137)
    Train: 275 | Test: 2476
    >>> VPN Category Baseline Weighted F1: 0.8149
                   precision    recall  f1-score   support

         VPN_Chat       0.67      0.4

In [7]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Same config as your main model for fair comparison
LABEL_PERCENTAGE = 0.30  # 20% labeled data

# --- 1. Data Loading & Feature Selection ---
def load_stats_only():
    print("--- Loading Merged CSV (Stats Only) ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # --- KEY STEP: SELECT ONLY STATS COLUMNS ---
    # We explicitly EXCLUDE 'alpha_' (Payload) and label/metadata columns
    exclude_keywords = ['alpha_', 'application', 'category', 'binary_type', 'filename']

    # Logic: Keep columns that start with beta/gamma/fft AND are not in exclude list
    stats_cols = [c for c in df.columns if
                  (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
                  and not any(x in c for x in exclude_keywords)]

    print(f"Selected {len(stats_cols)} Statistical Features (Beta/Gamma/FFT).")
    print(f"Skipped Alpha (Payload) features.")

    X_stats = df[stats_cols].values.astype('float32')

    # Normalize (Standard practice for XGBoost to converge faster, though not strictly required)
    print("Normalizing Statistical Features...")
    X_stats = StandardScaler().fit_transform(X_stats)

    return df, X_stats

# --- 2. Generic Trainer (Same as before) ---
def train_task(X, y, task_name):
    print(f"\n>>> Starting Baseline Task: {task_name}")
    print(f"    Data Shape: {X.shape}")

    # 1. Split (Same 20% split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # 2. Train (Same XGBoost Hyperparameters)
    # We use the exact same settings to ensure the difference comes from DATA, not the model.
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    # 3. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

# --- 3. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X_stats = load_stats_only()
    if df_raw is None: return

    # 2. Reconstruct Labels (VPN_Prefixing)
    # We need to rebuild the labels exactly as we did for the main model
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # The Input for all tasks is just the Stats
    X_final = X_stats

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 1: BINARY DETECTION (Stats Only)")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task(X_final, y_bin, "Binary Baseline")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))


    # ==========================================
    # EXPERIMENT 2: VPN CATEGORY
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 2: VPN CATEGORY (Stats Only)")
    print("="*40)

    mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final[mask]
    y_cat_raw = df_labels.loc[mask, 'Category']

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)
    y_test, y_pred = train_task(X_cat, y_cat, "VPN Category Baseline")
    print(classification_report(y_test, y_pred, target_names=le_cat.classes_))


    # ==========================================
    # EXPERIMENT 3: VPN TOP APPS
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 3: VPN TOP 6 APPS (Stats Only)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]
    mask = df_labels['Application'].isin(target_apps)
    X_app = X_final[mask]
    y_app_raw = df_labels.loc[mask, 'Application']

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)
    y_test, y_pred = train_task(X_app, y_app, "VPN Top Apps Baseline")
    print(classification_report(y_test, y_pred, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV (Stats Only) ---
Selected 137 Statistical Features (Beta/Gamma/FFT).
Skipped Alpha (Payload) features.
Normalizing Statistical Features...
Reconstructing Labels...

 BASELINE 1: BINARY DETECTION (Stats Only)

>>> Starting Baseline Task: Binary Baseline
    Data Shape: (12555, 137)
    Train: 3766 | Test: 8789
    >>> Binary Baseline Weighted F1: 0.9579
              precision    recall  f1-score   support

     Non-VPN       0.99      0.96      0.97      6863
         VPN       0.86      0.96      0.91      1926

    accuracy                           0.96      8789
   macro avg       0.92      0.96      0.94      8789
weighted avg       0.96      0.96      0.96      8789


 BASELINE 2: VPN CATEGORY (Stats Only)

>>> Starting Baseline Task: VPN Category Baseline
    Data Shape: (2751, 137)
    Train: 825 | Test: 1926
    >>> VPN Category Baseline Weighted F1: 0.9094
                   precision    recall  f1-score   support

         VPN_Chat       0.71      0.80

In [4]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
MERGED_CSV_PATH = os.path.join(BASE_PATH, "VNAT_merged_components_consistent.csv")

# Same config as your main model for fair comparison
LABEL_PERCENTAGE = 0.50  # 20% labeled data

# --- 1. Data Loading & Feature Selection ---
def load_stats_only():
    print("--- Loading Merged CSV (Stats Only) ---")
    if not os.path.exists(MERGED_CSV_PATH):
        print(f"Error: {MERGED_CSV_PATH} not found.")
        return None, None

    df = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # --- KEY STEP: SELECT ONLY STATS COLUMNS ---
    # We explicitly EXCLUDE 'alpha_' (Payload) and label/metadata columns
    exclude_keywords = ['alpha_', 'application', 'category', 'binary_type', 'filename']

    # Logic: Keep columns that start with beta/gamma/fft AND are not in exclude list
    stats_cols = [c for c in df.columns if
                  (c.startswith('beta_') or c.startswith('gamma_') or c.startswith('fft_'))
                  and not any(x in c for x in exclude_keywords)]

    print(f"Selected {len(stats_cols)} Statistical Features (Beta/Gamma/FFT).")
    print(f"Skipped Alpha (Payload) features.")

    X_stats = df[stats_cols].values.astype('float32')

    # Normalize (Standard practice for XGBoost to converge faster, though not strictly required)
    print("Normalizing Statistical Features...")
    X_stats = StandardScaler().fit_transform(X_stats)

    return df, X_stats

# --- 2. Generic Trainer (Same as before) ---
def train_task(X, y, task_name):
    print(f"\n>>> Starting Baseline Task: {task_name}")
    print(f"    Data Shape: {X.shape}")

    # 1. Split (Same 20% split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # 2. Train (Same XGBoost Hyperparameters)
    # We use the exact same settings to ensure the difference comes from DATA, not the model.
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    # 3. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

# --- 3. Main Execution ---
def main():
    # 1. Load Data
    df_raw, X_stats = load_stats_only()
    if df_raw is None: return

    # 2. Reconstruct Labels (VPN_Prefixing)
    # We need to rebuild the labels exactly as we did for the main model
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    final_traffic, final_category, final_app = [], [], []
    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix, t_type = "VPN", "VPN"
        else:
            prefix, t_type = "NonVPN", "Non-VPN"

        final_traffic.append(t_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # The Input for all tasks is just the Stats
    X_final = X_stats

    # ==========================================
    # EXPERIMENT 1: BINARY TASK
    # ==========================================
    print("\n" + "="*40)
    print(" BASELINE 1: BINARY DETECTION (Stats Only)")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])
    y_test, y_pred = train_task(X_final, y_bin, "Binary Baseline")
    print(classification_report(y_test, y_pred, target_names=le_bin.classes_))

if __name__ == "__main__":
    main()

--- Loading Merged CSV (Stats Only) ---
Selected 137 Statistical Features (Beta/Gamma/FFT).
Skipped Alpha (Payload) features.
Normalizing Statistical Features...
Reconstructing Labels...

 BASELINE 1: BINARY DETECTION (Stats Only)

>>> Starting Baseline Task: Binary Baseline
    Data Shape: (3709, 137)
    Train: 1854 | Test: 1855


XGBoostError: [12:39:43] /workspace/src/objective/regression_obj.cu:119: Check failed: is_valid: base_score must be in (0,1) for the logistic loss.
Stack trace:
  [bt] (0) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x2be09c) [0x7877090be09c]
  [bt] (1) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x1077f04) [0x787709e77f04]
  [bt] (2) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x1078525) [0x787709e78525]
  [bt] (3) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x6d20c9) [0x7877094d20c9]
  [bt] (4) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x6f53c7) [0x7877094f53c7]
  [bt] (5) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(XGBoosterUpdateOneIter+0x77) [0x787708fcab77]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x787753e05e2e]
  [bt] (7) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x787753e02493]
  [bt] (8) /usr/lib/python3.12/lib-dynload/_cty